# GCP Infra, DevOps & Reliability — Hands-On Lab
### GCP Training Program — Day 5, Module 8: GCP Infra, DevOps & Reliability

**What this notebook covers:** live Monitoring & Logging, LLM connectivity patterns from GCP
(including Claude via Vertex AI Model Garden's partner models), and guided walkthroughs of ML on
GKE and multi-project topology.

**Note on scope:** the agenda line "GCP connectivity with Claude / LLMs" is a bit open to
interpretation — I've read it as *how to securely call an external/partner LLM API from GCP
infrastructure* (credential handling via Secret Manager, and Claude's specific availability as a
Vertex AI Model Garden partner model). Adjust Section 5 if you had something more specific in mind.

## ⚠️ Cost & trial-account notes
**GKE is the one genuinely expensive resource on this agenda.** Even the smallest Autopilot cluster
takes 5-10 minutes to provision and bills continuously once created. Section 6 is a **guided
walkthrough with optional live execution** — read the cost note there before deciding whether to
run it in front of a trial-account class. Multi-project topology (Section 7) needs multiple
projects and org-level permissions most trial accounts don't have, so it's diagram/walkthrough
only, not executable here regardless.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs
against your own project.


## 1. Authenticate This Session

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")

## 2. Configure Your Project & Region

In [ ]:
import time
import os

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

_suffix = int(time.time())
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

!gcloud config set project {PROJECT_ID}
!gcloud services enable logging.googleapis.com monitoring.googleapis.com secretmanager.googleapis.com --project={PROJECT_ID}
print("Project configured:", PROJECT_ID)

## 3. Monitoring & Logging
### 3.1 Write & Read a Custom Log Entry
**What this does:** writes a structured log entry programmatically (the same mechanism any
application would use), then reads it back with a filter — the basic read/write loop behind every
more advanced logging feature.

In [ ]:
!pip install --quiet google-cloud-logging google-cloud-monitoring

In [ ]:
from google.cloud import logging as gcp_logging

logging_client = gcp_logging.Client(project=PROJECT_ID)
logger = logging_client.logger("training-demo-log")

logger.log_struct({
    "message": "Training demo log entry",
    "severity_context": "demo",
    "component": "gcp-infra-lab",
})
print("Wrote a structured log entry to 'training-demo-log'.")

In [ ]:
!gcloud logging read "logName=projects/{PROJECT_ID}/logs/training-demo-log" --project={PROJECT_ID} --limit=5 --format=json

### 3.2 Create a Log-Based Metric
**What this does:** turns a log filter into a numeric time-series metric — every future log entry
matching the filter increments a counter you can chart, alert on, or dashboard, instead of manually
grepping logs after the fact.

In [ ]:
METRIC_NAME = f"training-demo-metric-{_suffix}"

!gcloud logging metrics create {METRIC_NAME} \
    --project={PROJECT_ID} \
    --description="Counts training-demo-log entries" \
    --log-filter='logName="projects/{PROJECT_ID}/logs/training-demo-log"'

### 3.3 Create an Alert Policy on That Metric
**What this does:** wires the metric above to an alert condition — here, "fire if the count
exceeds 0 in a 60-second window," a deliberately trivial threshold so it's easy to trigger live for
demo purposes (a real policy would use a meaningful threshold and duration).

In [ ]:
import json

alert_policy = {
    "displayName": f"training-demo-alert-{_suffix}",
    "conditions": [
        {
            "displayName": "Log entries present",
            "conditionThreshold": {
                "filter": f'metric.type="logging.googleapis.com/user/{METRIC_NAME}" AND resource.type="global"',
                "comparison": "COMPARISON_GT",
                "thresholdValue": 0,
                "duration": "60s",
                "aggregations": [{"alignmentPeriod": "60s", "perSeriesAligner": "ALIGN_COUNT"}],
            },
        }
    ],
    "combiner": "OR",
    "notificationChannels": [],
}

with open("alert_policy.json", "w") as f:
    json.dump(alert_policy, f)

print("Wrote alert_policy.json — note: no notificationChannels configured, so this creates the")
print("policy without actually paging anyone. Add a real channel ID for a production policy.")

In [ ]:
!gcloud alpha monitoring policies create --project={PROJECT_ID} --policy-from-file=alert_policy.json

## 4. Secret Manager — Foundation for Section 5
### 4.1 Store an API Key Securely
**What this does:** demonstrates the pattern used to hold any external credential (an LLM provider
API key, a database password) without ever putting it in code or a notebook cell in plain text —
code references the *secret name*, and IAM controls who/what can read the actual value.

In [ ]:
SECRET_ID = f"demo-llm-api-key-{_suffix}"

api_key_value = input("Paste any placeholder value to store as a demo secret (not a real key needed): ").strip()

!echo -n "{api_key_value}" | gcloud secrets create {SECRET_ID} --project={PROJECT_ID} --data-file=- --replication-policy=automatic

### 4.2 Read the Secret Back Programmatically

In [ ]:
!pip install --quiet google-cloud-secret-manager

In [ ]:
from google.cloud import secretmanager

sm_client = secretmanager.SecretManagerServiceClient()
secret_path = f"projects/{PROJECT_ID}/secrets/{SECRET_ID}/versions/latest"

response = sm_client.access_secret_version(request={"name": secret_path})
retrieved_value = response.payload.data.decode("UTF-8")
print("Retrieved secret value:", retrieved_value)
print("\nIn a real application, this is exactly how you'd fetch an LLM provider API key at runtime")
print("instead of hardcoding it — combine with IAM (roles/secretmanager.secretAccessor) scoped to")
print("only the service account that needs it.")

## 5. GCP Connectivity With Claude & Other LLMs
### 5.1 Claude on Vertex AI Model Garden (Partner Model)
**What this is:** Anthropic's Claude models are available as a **partner model** in Vertex AI Model
Garden in supported regions — this means GCP-native billing, IAM, and networking apply, without
going through Anthropic's API directly.

**Caveat for the class:** partner model availability, supported regions, and the exact SDK/endpoint
pattern change over time — confirm current availability and syntax against GCP's live documentation
before presenting this as definitive. The shape below is illustrative of the pattern, not guaranteed
to match the exact current API without a docs check.

In [ ]:
# Illustrative pattern for calling a Claude model via Vertex AI Model Garden partner models.
# Confirm exact model IDs/region availability against current Vertex AI documentation before
# using this live — partner model surfaces are one of the faster-evolving areas of the platform.

# !pip install --quiet "anthropic[vertex]"
#
# from anthropic import AnthropicVertex
#
# client = AnthropicVertex(project_id=PROJECT_ID, region=REGION)
# message = client.messages.create(
#     model="claude-sonnet-4@20250514",  # confirm the current partner-model ID string in docs
#     max_tokens=256,
#     messages=[{"role": "user", "content": "Explain GCP IAM in two sentences."}],
# )
# print(message.content)

print("Reference pattern above — verify current model IDs/region support before running live.")

### 5.2 The General Connectivity Pattern (Applies to Any External LLM Provider)
Whether calling Claude via Model Garden, Anthropic's own API directly, or any other external model
provider from GCP infrastructure, the same three concerns apply — worth stating explicitly to the
class as the reusable pattern, independent of which specific provider they end up using:

1. **Credential storage** — Secret Manager (Section 4), never hardcoded, never committed to a repo.
2. **Network egress** — if calling an external API (not a GCP-native partner model endpoint) from a
   VPC with restricted egress, a Cloud NAT or explicit firewall egress rule is needed; fully private
   GKE/Compute workloads with no NAT cannot reach the public internet at all by default.
3. **IAM scoping** — grant the calling service account only `roles/secretmanager.secretAccessor` on
   the specific secret, not project-wide Secret Manager access — least privilege, same principle as
   Day 3's security module.

In [ ]:
sm_client.delete_secret(request={"name": f"projects/{PROJECT_ID}/secrets/{SECRET_ID}"})
print("Deleted demo secret — cleaning up now since Section 5 only needed it as an illustration.")

## 6. ML on GKE (Guided Walkthrough — Optional Live Execution)
**Cost note, read before deciding whether to run this live:** even the smallest GKE Autopilot
cluster takes 5-10 minutes to provision and bills continuously from the moment it's created —
there is no smaller "trial-safe" tier the way there is for a single BigQuery query or Cloud
Storage bucket. **Recommendation: walk through the commands and concept without running them live
in a trial-account class**, unless you've budgeted specifically for it and are ready to delete the
cluster within a few minutes of creating it. Cloud Build (Section 6.5-6.7) bills separately and
far more cheaply — a few build-minutes per pipeline run — so that part is comfortable to actually
trigger live even on a trial account.

**The concept:** GKE Autopilot removes node-management entirely (Google manages node provisioning,
you only think in terms of pods/deployments) — the natural choice for a training demo *if* you do
run it live, since there's no node pool sizing decision to explain or get wrong.

**What this section demonstrates, end to end:**
1. Deploy a small web app, **exposed with a real public URL** (`Service` of type `LoadBalancer`) —
   something you can actually open in a browser and show the class, not just `kubectl` output.
2. A **real Cloud Build pipeline** that performs the release — triggered manually here to simulate
   what a CI/CD system would do automatically on every merge, but it's the same `gcloud builds
   submit` call a Git-push trigger would fire.
3. A **rolling update** from v1 to v2 via that pipeline, watched live with `kubectl rollout status`.
4. The **same cluster, same pipeline pattern**, reused for a second, genuinely ML workload — a
   small model-serving container — to show this isn't a one-off toy, it's a repeatable pattern.

In [ ]:
CLUSTER_NAME = f"training-demo-cluster-{_suffix}"

# OPTIONAL — only uncomment and run if you've decided to demo a real cluster live.
# Autopilot auto-sizes nodes; there is no smaller manual machine-type choice to make here.

# !gcloud container clusters create-auto {CLUSTER_NAME} \
#     --project={PROJECT_ID} \
#     --region={REGION}

print("Cluster creation intentionally commented out — see the cost note above before uncommenting.")

### 6.2 Connect `kubectl` to the Cluster
**Only run this if you created the cluster above.** `get-credentials` fetches the cluster's
connection info and writes it to your local `kubeconfig`, so subsequent `kubectl` commands in this
notebook target the right cluster automatically.

In [ ]:
# OPTIONAL — only run if the cluster above is live.
# !gcloud container clusters get-credentials {CLUSTER_NAME} --region={REGION} --project={PROJECT_ID}
# !kubectl get nodes

print("Cell intentionally commented out — uncomment once the cluster in 6.1 is Ready.")

### 6.3 Deploy Version 1 — Exposed as a Real Web App
Two manifests this time: the `Deployment` (the running container, same pattern as before) plus a
`Service` of type `LoadBalancer` — this is what actually gives you a public URL. GKE provisions a
real external IP for it, which takes 1-2 minutes to appear after the Service is created.

`gcr.io/google-samples/hello-app:1.0` is Google's own public sample image used in official GKE
tutorials for exactly this kind of version-bump demo — it serves a simple page showing its own
version number, which is what makes the v1 → v2 switch visually obvious in a browser.

In [ ]:
deployment_v1_yaml = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: demo-model-server
spec:
  replicas: 2
  selector:
    matchLabels:
      app: demo-model-server
  template:
    metadata:
      labels:
        app: demo-model-server
    spec:
      containers:
        - name: demo-model-server
          image: gcr.io/google-samples/hello-app:1.0
          ports:
            - containerPort: 8080
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
"""

service_yaml = """\
apiVersion: v1
kind: Service
metadata:
  name: demo-model-server-svc
spec:
  type: LoadBalancer
  selector:
    app: demo-model-server
  ports:
    - port: 80
      targetPort: 8080
"""

with open("deployment-v1.yaml", "w") as f:
    f.write(deployment_v1_yaml)
with open("service.yaml", "w") as f:
    f.write(service_yaml)

print(deployment_v1_yaml)
print(service_yaml)

# OPTIONAL — only run if kubectl is connected (Section 6.2).
# !kubectl apply -f deployment-v1.yaml
# !kubectl apply -f service.yaml
# !kubectl rollout status deployment/demo-model-server --timeout=180s

print("Apply commands intentionally commented out — uncomment once kubectl is connected.")

### 6.4 Get the Public URL & Open It
`EXTERNAL-IP` starts as `<pending>` for a minute or two while GKE provisions the load balancer —
this loop polls until it's ready, then prints a URL you can literally open in a browser and show
the class.

In [ ]:
# OPTIONAL — only run once the Service from 6.3 has been applied.

# import time as _time
# for _ in range(30):
#     ip = !kubectl get service demo-model-server-svc -o jsonpath="{.status.loadBalancer.ingress[0].ip}"
#     ip = ip[0].strip() if ip else ""
#     if ip:
#         print(f"App is live at: http://{ip}")
#         break
#     print("Waiting for external IP...")
#     _time.sleep(10)

print("Cell intentionally commented out — uncomment once the Service in 6.3 is applied.")

### 6.5 Set Up the Release Pipeline (Cloud Build)
This is a genuine, reusable Cloud Build pipeline — not just a reference file this time. It takes
the deployment name, container name, target image, and cluster location as **substitution
variables**, so the exact same pipeline definition works for the hello-app release below *and* the
ML workload in Section 6.8 — one pipeline, two different workloads, which is the actual point of
a CI/CD pipeline being a reusable asset rather than a one-off script per app.

`gcr.io/cloud-builders/kubectl` is Google's official Cloud Build builder image for `kubectl` — it
reads `CLOUDSDK_COMPUTE_REGION`/`CLOUDSDK_CONTAINER_CLUSTER` env vars to auto-fetch cluster
credentials, so no separate `get-credentials` step is needed inside the pipeline itself.

In [ ]:
import os

os.makedirs("gke_pipeline", exist_ok=True)

cloudbuild_release_yaml = """\
steps:
  - name: "gcr.io/cloud-builders/kubectl"
    args: ["set", "image", "deployment/$_DEPLOYMENT_NAME", "$_CONTAINER_NAME=$_IMAGE"]
    env:
      - "CLOUDSDK_COMPUTE_REGION=$_REGION"
      - "CLOUDSDK_CONTAINER_CLUSTER=$_CLUSTER_NAME"
  - name: "gcr.io/cloud-builders/kubectl"
    args: ["rollout", "status", "deployment/$_DEPLOYMENT_NAME", "--timeout=180s"]
    env:
      - "CLOUDSDK_COMPUTE_REGION=$_REGION"
      - "CLOUDSDK_CONTAINER_CLUSTER=$_CLUSTER_NAME"
options:
  logging: CLOUD_LOGGING_ONLY
"""

with open("gke_pipeline/cloudbuild-release.yaml", "w") as f:
    f.write(cloudbuild_release_yaml)

print(cloudbuild_release_yaml)

# OPTIONAL — grants Cloud Build's default service account permission to deploy to GKE.
# Recent GCP projects restrict this by default, so this step is easy to forget and a common
# reason a first pipeline run fails with a permissions error.
# project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
# project_number = project_number[0].strip()
# cloudbuild_sa = f"serviceAccount:{project_number}@cloudbuild.gserviceaccount.com"
# !gcloud projects add-iam-policy-binding {PROJECT_ID} \
#     --member="{cloudbuild_sa}" \
#     --role="roles/container.developer"

print("IAM grant intentionally commented out — uncomment once you're ready to trigger the pipeline.")

### 6.6 Trigger the Pipeline to "Release" Version 2
**This is the CI/CD moment.** `gcloud builds submit` here is doing exactly what a Git-push trigger
would do automatically in a real setup — the only difference is a human is running the command
instead of a webhook. Watch the Cloud Build logs stream the `kubectl set image` and
`kubectl rollout status` steps executing, then reload the URL from Section 6.4 — the page content
changes from v1 to v2 because GKE performed a **rolling update**, replacing pods gradually rather
than taking the app down.

In [ ]:
# OPTIONAL — only run once 6.3-6.5 are live.

# !gcloud builds submit gke_pipeline/ \
#     --config=gke_pipeline/cloudbuild-release.yaml \
#     --substitutions=_DEPLOYMENT_NAME=demo-model-server,_CONTAINER_NAME=demo-model-server,_IMAGE=gcr.io/google-samples/hello-app:2.0,_REGION={REGION},_CLUSTER_NAME={CLUSTER_NAME}

print("Pipeline trigger intentionally commented out — uncomment once ready to release v2 live.")
print("\nAfter this completes, reload the URL from Section 6.4 — the page now shows version 2.0.0.")

### 6.7 Rollback Demo (Bonus)
Two ways to roll back, both worth showing: the direct `kubectl` command (fast, but bypasses your
pipeline/audit trail), and re-triggering the *same pipeline* with the old image tag (slower, but
goes through the same reviewable process as any other release — the approach a real team would
actually want).

In [ ]:
# OPTIONAL — direct rollback, bypasses the pipeline:
# !kubectl rollout undo deployment/demo-model-server
# !kubectl rollout status deployment/demo-model-server --timeout=180s

# OPTIONAL — pipeline-driven rollback, goes through the same process as any other release:
# !gcloud builds submit gke_pipeline/ \
#     --config=gke_pipeline/cloudbuild-release.yaml \
#     --substitutions=_DEPLOYMENT_NAME=demo-model-server,_CONTAINER_NAME=demo-model-server,_IMAGE=gcr.io/google-samples/hello-app:1.0,_REGION={REGION},_CLUSTER_NAME={CLUSTER_NAME}

print("Rollback commands intentionally commented out — uncomment whichever approach you want to demo.")

## 6.8 ML Use Case on GKE — a Dedicated CI/CD Pipeline
Sections 6.3-6.7 demonstrated CI/CD with a generic web app: the pipeline only ever had to *switch
an image tag* that already existed. A real ML release is usually one step earlier in the chain —
**the new version doesn't exist as an image yet**; retraining or changing a hyperparameter produces
code that has to be built into a new container *before* anything can be deployed. So this section
uses its own, slightly larger pipeline: **build (with a new hyperparameter baked in) → push →
deploy → wait for rollout**, all in one command, versus the switch-only pipeline from before.

**The easy-to-explain concept:** the "new model version" here is just a different
`max_depth` hyperparameter for a RandomForest classifier — a one-line change a data scientist would
actually make, wired through as a Docker build argument so releasing it needs **zero manual file
edits**, just a different value passed to one command.
### 6.8.1 Write the App Code, Dockerfile & Requirements
The Dockerfile now declares `MODEL_VERSION` and `MAX_DEPTH` as build **arguments** (`ARG`), not
fixed values — each pipeline run can bake in a different value at build time without touching this
file.

In [ ]:
import os

os.makedirs("ml_app", exist_ok=True)

app_py = """\
import os
from flask import Flask, jsonify
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

app = Flask(__name__)

MODEL_VERSION = os.environ.get("MODEL_VERSION", "v1")
MAX_DEPTH = int(os.environ.get("MAX_DEPTH", "4"))

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=50, max_depth=MAX_DEPTH, random_state=42).fit(X, y)

@app.route("/")
def home():
    return f"Iris classifier serving on GKE — model {MODEL_VERSION} (max_depth={MAX_DEPTH})\\n"

@app.route("/predict")
def predict():
    sample = [[5.1, 3.5, 1.4, 0.2]]  # fixed sample Iris measurement, for a repeatable demo
    prediction = int(model.predict(sample)[0])
    return jsonify({"model_version": MODEL_VERSION, "max_depth": MAX_DEPTH, "prediction": prediction})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)
"""

dockerfile = """\
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .

ARG MODEL_VERSION=v1
ARG MAX_DEPTH=4
ENV MODEL_VERSION=${MODEL_VERSION}
ENV MAX_DEPTH=${MAX_DEPTH}

EXPOSE 8080
CMD ["python", "app.py"]
"""

requirements_txt = "flask\nscikit-learn\n"

with open("ml_app/app.py", "w") as f:
    f.write(app_py)
with open("ml_app/Dockerfile", "w") as f:
    f.write(dockerfile)
with open("ml_app/requirements.txt", "w") as f:
    f.write(requirements_txt)

print("Wrote ml_app/app.py, Dockerfile (with MODEL_VERSION/MAX_DEPTH as build ARGs), requirements.txt")

### 6.8.2 Write the ML Release Pipeline (Build + Push + Deploy, One Command)
This is a **separate, dedicated pipeline** from Section 6.5's — deliberately, since an ML release
has an extra stage a plain tag-switch doesn't. Every value that changes between releases
(`_VERSION_TAG`, `_MAX_DEPTH`, `_IMAGE`) is a **substitution**, so releasing v2, v3, v4... is always
the same pipeline, run with different `--substitutions` — never a hand-edited file.

**Step-by-step:** (1) Docker builds the image, baking in `MAX_DEPTH`/`MODEL_VERSION` as build args;
(2) push the built image to Artifact Registry; (3) point the live Deployment at the new image with
`kubectl set image`; (4) wait for the rolling update to finish.

In [ ]:
os.makedirs("gke_pipeline", exist_ok=True)

cloudbuild_ml_release_yaml = """\
steps:
  - name: "gcr.io/cloud-builders/docker"
    args:
      - "build"
      - "--build-arg"
      - "MODEL_VERSION=$_VERSION_TAG"
      - "--build-arg"
      - "MAX_DEPTH=$_MAX_DEPTH"
      - "-t"
      - "$_IMAGE"
      - "."
  - name: "gcr.io/cloud-builders/docker"
    args: ["push", "$_IMAGE"]
  - name: "gcr.io/cloud-builders/kubectl"
    args: ["set", "image", "deployment/$_DEPLOYMENT_NAME", "$_CONTAINER_NAME=$_IMAGE"]
    env:
      - "CLOUDSDK_COMPUTE_REGION=$_REGION"
      - "CLOUDSDK_CONTAINER_CLUSTER=$_CLUSTER_NAME"
  - name: "gcr.io/cloud-builders/kubectl"
    args: ["rollout", "status", "deployment/$_DEPLOYMENT_NAME", "--timeout=180s"]
    env:
      - "CLOUDSDK_COMPUTE_REGION=$_REGION"
      - "CLOUDSDK_CONTAINER_CLUSTER=$_CLUSTER_NAME"
images:
  - "$_IMAGE"
options:
  logging: CLOUD_LOGGING_ONLY
"""

with open("ml_app/cloudbuild-ml-release.yaml", "w") as f:
    f.write(cloudbuild_ml_release_yaml)

print(cloudbuild_ml_release_yaml)

### 6.8.3 Bootstrap: First Deploy (v1)
**Why this step exists outside the pipeline:** `kubectl set image` (used by the pipeline above)
updates an *existing* Deployment — it has nothing to update the very first time, since no
Deployment exists yet. This mirrors a real project's day-one setup: the first release is a manual
bootstrap; every release *after* that goes through the pipeline (Section 6.8.4 onward). The
Dockerfile's `ARG` defaults (`MODEL_VERSION=v1`, `MAX_DEPTH=4`) mean this bootstrap build needs no
special flags at all.

In [ ]:
ML_REPO = "gke-ml-demo"
ML_IMAGE_V1 = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{ML_REPO}/iris-server:v1"

# OPTIONAL — creates the Artifact Registry repo and builds+pushes the bootstrap v1 image.

# !gcloud artifacts repositories create {ML_REPO} --repository-format=docker --location={REGION}
# !gcloud builds submit ml_app/ --tag {ML_IMAGE_V1}

ml_deployment_yaml = f"""\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-ml-server
spec:
  replicas: 2
  selector:
    matchLabels:
      app: iris-ml-server
  template:
    metadata:
      labels:
        app: iris-ml-server
    spec:
      containers:
        - name: iris-ml-server
          image: {ML_IMAGE_V1}
          ports:
            - containerPort: 8080
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
"""

ml_service_yaml = """\
apiVersion: v1
kind: Service
metadata:
  name: iris-ml-server-svc
spec:
  type: LoadBalancer
  selector:
    app: iris-ml-server
  ports:
    - port: 80
      targetPort: 8080
"""

with open("ml-deployment-v1.yaml", "w") as f:
    f.write(ml_deployment_yaml)
with open("ml-service.yaml", "w") as f:
    f.write(ml_service_yaml)

print(ml_deployment_yaml)
print(ml_service_yaml)

# !kubectl apply -f ml-deployment-v1.yaml
# !kubectl apply -f ml-service.yaml
# !kubectl rollout status deployment/iris-ml-server --timeout=180s

print("Bootstrap commands intentionally commented out — uncomment once ready to run this live.")

### 6.8.4 Get the Prediction Endpoint URL & Call It
Same external-IP polling pattern as Section 6.4, this time for the ML service. Once you have the
IP, `curl http://IP/predict` returns a live JSON prediction — genuinely serving a model from GKE.

In [ ]:
# OPTIONAL — only run once the Service from 6.8.3 has been applied.

# import time as _time
# for _ in range(30):
#     ml_ip = !kubectl get service iris-ml-server-svc -o jsonpath="{.status.loadBalancer.ingress[0].ip}"
#     ml_ip = ml_ip[0].strip() if ml_ip else ""
#     if ml_ip:
#         print(f"ML service is live at: http://{ml_ip}/predict")
#         break
#     print("Waiting for external IP...")
#     _time.sleep(10)
#
# !curl -s http://{ml_ip}/predict

print("Cell intentionally commented out — uncomment once the ML Service in 6.8.3 is applied.")

### 6.8.5 Release Version 2 — One Command, No Manual Edits
**This is the full ML CI/CD flow.** `MAX_DEPTH=8` here stands in for "the ML team retrained with a
different hyperparameter" — in a real pipeline this would be triggered by a merge to `main` after a
training script or config change; here it's the same `gcloud builds submit` call, just run by hand.
Nothing about `ml_app/` is edited between v1 and v2 — the new value flows in entirely through
`--substitutions`. After this completes, calling `/predict` again shows `max_depth` change from 4 to
8 in the response — visible proof the new version is live.

In [ ]:
ML_IMAGE_V2 = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{ML_REPO}/iris-server:v2"

# OPTIONAL — one command: builds v2 (MAX_DEPTH=8) with build-args, pushes it, deploys it, and
# waits for the rollout — no Dockerfile edits, no separate deploy step.

# !gcloud builds submit ml_app/ \
#     --config=ml_app/cloudbuild-ml-release.yaml \
#     --substitutions=_VERSION_TAG=v2,_MAX_DEPTH=8,_IMAGE={ML_IMAGE_V2},_DEPLOYMENT_NAME=iris-ml-server,_CONTAINER_NAME=iris-ml-server,_REGION={REGION},_CLUSTER_NAME={CLUSTER_NAME}

# !curl -s http://{ml_ip}/predict

print("Release command intentionally commented out — uncomment once ready to demo v2 of the ML service.")
print("Expect the /predict response's max_depth field to change from 4 to 8 once this completes.")

### 6.8.6 Rollback Demo (Bonus)
Same pipeline, pointed back at the v1 image and hyperparameter — a real rollback, not just a
conceptual one, and still zero manual file edits.

In [ ]:
# OPTIONAL — rolls back to v1 through the same pipeline.

# !gcloud builds submit ml_app/ \
#     --config=ml_app/cloudbuild-ml-release.yaml \
#     --substitutions=_VERSION_TAG=v1,_MAX_DEPTH=4,_IMAGE={ML_IMAGE_V1},_DEPLOYMENT_NAME=iris-ml-server,_CONTAINER_NAME=iris-ml-server,_REGION={REGION},_CLUSTER_NAME={CLUSTER_NAME}

# !curl -s http://{ml_ip}/predict

print("Rollback command intentionally commented out — uncomment to demo reverting to v1.")

### 6.9 Cleanup
Delete both workloads first (web app + ML service), then both Artifact Registry repos, then the
cluster last — run this **immediately** after the demo, not at the end of the day. The cluster is
the expensive part and bills continuously until deleted; Cloud Build itself has nothing left
running once a build finishes, so there's nothing to clean up there beyond the images it produced.

In [ ]:
# OPTIONAL — matches whichever cells above you actually ran live.

# Delete both workloads (web app from 6.3, ML service from 6.8)
# !kubectl delete -f service.yaml --ignore-not-found
# !kubectl delete -f deployment-v1.yaml --ignore-not-found
# !kubectl delete -f ml-service.yaml --ignore-not-found
# !kubectl delete -f ml-deployment-v1.yaml --ignore-not-found

# Delete the Artifact Registry repo (all image tags — v1, v2, etc. — go with it)
# !gcloud artifacts repositories delete {ML_REPO} --location={REGION} --quiet

# Delete the cluster — this is the resource that actually bills continuously
# !gcloud container clusters delete {CLUSTER_NAME} --project={PROJECT_ID} --region={REGION} --quiet

print("Cleanup commands intentionally commented out — matches whichever optional cells above you ran.")
print("If you created a real cluster in Section 6.1, uncomment the cluster delete line above at minimum.")

## 7. Multi-Project Topology (Guided Walkthrough Only)
**Why this is walkthrough-only, not executable in this notebook:** Shared VPC and VPC Service
Controls both require multiple projects and org-level (or folder-level) IAM permissions that a
single trial-account project doesn't have — there's nothing meaningful to run here regardless of
budget. Present this as architecture/diagram content:

- **Shared VPC** — one "host" project owns the VPC network; other "service" projects attach to it
  and deploy resources using its subnets, while keeping their own billing, IAM, and resource
  ownership separate. Common pattern: a central networking team owns the host project; individual
  product teams own service projects.
- **VPC Service Controls** — draws a security perimeter *around* a set of projects/services (e.g.
  "BigQuery and GCS in these 3 projects"), blocking data exfiltration even by someone with valid
  IAM credentials, if they're accessing from outside the defined perimeter (e.g. a personal Gmail
  session vs. a corporate-network session).
- **When to bring this up relative to Day 3's security module:** IAM controls *who* can do *what*;
  Shared VPC controls *network topology*; VPC Service Controls adds a perimeter *on top of* both —
  worth being explicit that these are layered, not alternatives to each other.